In [2]:
import pandas as pd

In [3]:
#Load
data_5min2016 = pd.read_csv('resampled_5min_2016_with_features.csv')
data_5min2017 = pd.read_csv('resampled_5min_2017_with_features.csv')
data_5min2018 = pd.read_csv('resampled_5min_2018_with_features.csv')
data_5min2019 = pd.read_csv('resampled_5min_2019_with_features.csv')
data_5min2020 = pd.read_csv('resampled_5min_2020_with_features.csv')
data_5min2021 = pd.read_csv('resampled_5min_2021_with_features.csv')
data_5min2022 = pd.read_csv('resampled_5min_2022_with_features.csv')
data_5min2023 = pd.read_csv('resampled_5min_2023_with_features.csv')

In [4]:
!pip install ta
import ta

for year_data in [data_5min2016, data_5min2017, data_5min2018, data_5min2019, data_5min2020, data_5min2021, data_5min2022]:
    # Simple Moving Averages
    year_data['SMA_5'] = ta.trend.sma_indicator(year_data['Mid_Price_1_last'], window=5)
    year_data['SMA_20'] = ta.trend.sma_indicator(year_data['Mid_Price_1_last'], window=20)

    # Exponential Moving Averages
    year_data['EMA_5'] = ta.trend.ema_indicator(year_data['Mid_Price_1_last'], window=5)
    year_data['EMA_20'] = ta.trend.ema_indicator(year_data['Mid_Price_1_last'], window=20)

    # RSI
    year_data['RSI'] = ta.momentum.rsi(year_data['Mid_Price_1_last'], window=14)

    # MACD
    year_data['MACD'] = ta.trend.macd_diff(year_data['Mid_Price_1_last'], window_slow=26, window_fast=12, window_sign=9)

    # Bollinger Bands
    bollinger = ta.volatility.BollingerBands(close=year_data['Mid_Price_1_last'], window=20, window_dev=2)
    year_data['Bollinger_High'] = bollinger.bollinger_hband()
    year_data['Bollinger_Low'] = bollinger.bollinger_lband()

    # ATR
    year_data['ATR'] = ta.volatility.average_true_range(
        high=year_data['Mid_Price_1_max'], 
        low=year_data['Mid_Price_1_min'], 
        close=year_data['Mid_Price_1_last'], 
        window=14
    )


for year_data in [data_5min2016, data_5min2017, data_5min2018, data_5min2019, data_5min2020, data_5min2021, data_5min2022]:
    year_data.dropna(inplace=True)


In [5]:
# X and y 
for year_data in [data_5min2016, data_5min2017, data_5min2018, data_5min2019, data_5min2020, data_5min2021, data_5min2022]:
    year_data['Direction'] = year_data['Mid_Price_1_last'].diff().apply(lambda x: 1 if x > 0 else -1).shift(-1)
    year_data.dropna(inplace=True)  # Remove nans

    X = year_data.drop(columns=['Mid_Price_1_last', 'Time (sec)', 'Date', 'Direction'])
    y = year_data['Direction']  

In [7]:
print(y.value_counts())

 1.0    9198
-1.0    9112
Name: Direction, dtype: int64


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib
import time

# Grid
param_grid = {
    'C': [0.1, 1, 10],               
    'kernel': ['rbf'],                
    'gamma': ['scale', 0.1, 1, 10]        
}

start_time = time.time()

# 2016-2022
data_years = [data_5min2016, data_5min2017, data_5min2018, data_5min2019, data_5min2020, data_5min2021, data_5min2022]

# Loop through rolling window 
for i in range(len(data_years) - 1):
    # Train on year 1 to year i, test on year i+1
    train_data = pd.concat(data_years[:i+1])
    test_data = data_years[i+1]
    
    
    X_train = train_data.drop(columns=['Mid_Price_1_last', 'Time (sec)', 'Date', 'Direction'])
    y_train = train_data['Direction']
    
   
    X_test = test_data.drop(columns=['Mid_Price_1_last', 'Time (sec)', 'Date', 'Direction'])
    y_test = test_data['Direction']
    
    # Standardize data 
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)  
    X_test_scaled = scaler.transform(X_test)  
    
    # SVM classifier
    svc = SVC(random_state=10)
    
    # TimeSeriesSplit 
    tscv = TimeSeriesSplit(n_splits=6)
    
    # GridSearchCV for hyperparameter tuning on SVM
    grid_search_svm = GridSearchCV(estimator=svc, param_grid=param_grid, cv=tscv, n_jobs=-1, verbose=2)
    grid_search_svm.fit(X_train_scaled, y_train)

    
    # Best SVM model
    best_svm = grid_search_svm.best_estimator_
    
    # Save best model 
    model_filename_svm = f'best_svm_model_{i+1}_to_{i+2}.pkl'  
    joblib.dump(best_svm, model_filename_svm)
    
    # Make predictions on the test set with the best SVM model
    y_pred_svm = best_svm.predict(X_test_scaled)
    
    # Evaluate the model performance
    accuracy = accuracy_score(y_test, y_pred_svm)
    precision = precision_score(y_test, y_pred_svm, average='weighted')
    recall = recall_score(y_test, y_pred_svm, average='weighted')
    f1 = f1_score(y_test, y_pred_svm, average='weighted')
    
    # Display results 
    print(f"Year {i+1} -> {i+2}")
    print(f"Best SVM Parameters: {grid_search_svm.best_params_}")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall: {recall * 100:.2f}%")
    print(f"F1 Score: {f1 * 100:.2f}%")
    print("-" * 30)

# End time and display total time
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Total Time taken: {elapsed_time:.2f} seconds")

Fitting 6 folds for each of 12 candidates, totalling 72 fits


/opt/anaconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1245: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Year 1 -> 2
Best SVM Parameters: {'C': 1, 'gamma': 1, 'kernel': 'rbf'}
Accuracy: 50.67%
Precision: 25.67%
Recall: 50.67%
F1 Score: 34.08%
------------------------------
Fitting 6 folds for each of 12 candidates, totalling 72 fits
Year 2 -> 3
Best SVM Parameters: {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
Accuracy: 50.12%
Precision: 25.12%
Recall: 50.12%
F1 Score: 33.47%
------------------------------
Fitting 6 folds for each of 12 candidates, totalling 72 fits


/opt/anaconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1245: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Year 3 -> 4
Best SVM Parameters: {'C': 0.1, 'gamma': 0.1, 'kernel': 'rbf'}
Accuracy: 49.95%
Precision: 24.95%
Recall: 49.95%
F1 Score: 33.28%
------------------------------
Fitting 6 folds for each of 12 candidates, totalling 72 fits


/opt/anaconda3/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1245: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Year 4 -> 5
Best SVM Parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
Accuracy: 50.75%
Precision: 50.74%
Recall: 50.75%
F1 Score: 48.48%
------------------------------
Fitting 6 folds for each of 12 candidates, totalling 72 fits
Year 5 -> 6
Best SVM Parameters: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
Accuracy: 49.03%
Precision: 50.75%
Recall: 49.03%
F1 Score: 34.53%
------------------------------
Fitting 6 folds for each of 12 candidates, totalling 72 fits
Year 6 -> 7
Best SVM Parameters: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
Accuracy: 50.32%
Precision: 50.91%
Recall: 50.32%
F1 Score: 35.93%
------------------------------
Total Time taken: 34989.80 seconds
